<a href="https://colab.research.google.com/github/Exenoria/Entrenamiento-de-Nitshio/blob/main/Nitshio_QLoRA_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nitshio — QLoRA en Colab (versión a prueba de reinicios)

Entrena **solo en la nube**. En tu PC solo se usa Ollama para inferencia.

## Cómo usarlo
1. Activa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.
2. Sube `Nitshio.zip` a `/content` (panel Archivos → Subir). Debe ser el zip **nuevo**.
3. Ejecuta **todas las celdas en orden, de la 1 a la última**. Si reinicias la sesión, repite desde la celda 1: las variables se pierden, pero los paquetes y los archivos de `/content` se conservan.
4. Si Colab te desconecta y elimina el entorno, se pierde todo: hay que subir el zip otra vez.

## Qué corrige respecto al notebook anterior
- No fuerza versiones con `--force-reinstall` (rompía el entorno).
- Quita `jax`/`flax` solo si chocan con `numpy` (causa del falso error `AutoProcessor`).
- Aplica el parche del `SFTTrainer` para TRL reciente (`processing_class`, `dataset_text_field` en `SFTConfig`).
- No vuelve a extraer el zip si el proyecto ya está (así no pisa el parche).
- Muestra el entrenamiento en vivo y, si falla, la causa real.


In [1]:
# 1. GPU + ubicar o descomprimir el proyecto (no pisa nada si ya existe)
import os, subprocess, sys, zipfile, shutil
from pathlib import Path

def run(cmd, cwd=None, tail=40):
    # Ejecuta un comando mostrando la salida en vivo; si falla, muestra el final y lanza error.
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    code = proc.wait()
    if code != 0:
        print("\n--- ÚLTIMAS LÍNEAS ---")
        print("".join(lines[-tail:]))
        raise RuntimeError(f"Falló (código {code}): {' '.join(cmd[:4])}")

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "")
assert gpu.returncode == 0, "No hay GPU. Entorno de ejecución → Cambiar tipo → T4 GPU"

WORK_DIR = Path(os.environ.get("NITSHIO_WORKDIR", "/content"))
PROJECT_DIR = Path(os.environ.get("NITSHIO_DIR", str(WORK_DIR / "Nitshio")))

def _tiene_proyecto(p):
    return (p / "config" / "config.yaml").exists() and (p / "training").is_dir()

if _tiene_proyecto(PROJECT_DIR):
    print("Proyecto ya presente en", PROJECT_DIR)
elif _tiene_proyecto(WORK_DIR):
    PROJECT_DIR = WORK_DIR
    print("Proyecto ya presente en", PROJECT_DIR)
else:
    zips = [z for z in sorted(WORK_DIR.glob("*.zip")) if not z.name.startswith("nitshio-entrega")]
    assert zips, f"Sube Nitshio.zip a {WORK_DIR} (panel Archivos → Subir)"
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    for z in zips:
        print("Descomprimiendo", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(PROJECT_DIR)
    if not _tiene_proyecto(PROJECT_DIR):
        hijos = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and _tiene_proyecto(p)]
        assert hijos, "El zip no contiene config/config.yaml y training/"
        PROJECT_DIR = hijos[0]

os.chdir(PROJECT_DIR)
print("PROJECT_DIR =", PROJECT_DIR)
print("Directorio actual:", os.getcwd())


Tue Nov 10 20:10:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Entorno: instalar solo si faltan paquetes y verificar que trl/transformers importan
import importlib.util
from importlib import metadata

def _ok(snippet):
    r = subprocess.run([sys.executable, "-c", snippet], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or "")[-1500:]

faltan = [m for m in ("unsloth", "trl", "peft", "transformers", "bitsandbytes")
          if importlib.util.find_spec(m) is None]
if faltan:
    print("Faltan paquetes:", faltan, "-> instalando requirements-train.txt")
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"])
    raise RuntimeError(
        "Instalación terminada. REINICIA la sesión (Entorno de ejecución → Reiniciar sesión) "
        "y ejecuta de nuevo TODAS las celdas desde la 1."
    )

PRUEBA = "from transformers import AutoProcessor; from trl import SFTTrainer"
ok, err = _ok(PRUEBA)
if not ok:
    print("El import falló. Probable choque jax/numpy; quitando jax/jaxlib/flax...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib", "flax"])
    ok, err = _ok(PRUEBA)
if not ok:
    print(err)
    raise RuntimeError("Sigue fallando el import de trl/transformers. Copia el error de arriba.")

for nombre in ("transformers", "trl", "peft", "unsloth", "torch", "numpy"):
    try:
        print(f"{nombre:14s}", metadata.version(nombre))
    except metadata.PackageNotFoundError:
        print(f"{nombre:14s}", "no instalado")
print("Entorno OK: AutoProcessor y SFTTrainer importan bien.")


transformers   4.57.6
trl            0.29.1
peft           0.21.2
unsloth        2026.11.3
torch          2.14.1+cu130
numpy          2.4.6
Entorno OK: AutoProcessor y SFTTrainer importan bien.


In [3]:
# 3. Parche para TRL reciente (idempotente: se puede ejecutar varias veces)
p = PROJECT_DIR / "training" / "finetune_qlora.py"
t = p.read_text(encoding="utf-8")
cambios = []

if "processing_class" not in t:
    viejo = 'base["args"] = sft_args'
    nuevo = ('sft_args.dataset_text_field = "text"\n'
             '            base.pop("dataset_text_field", None)\n'
             '            base["processing_class"] = base.pop("tokenizer")\n'
             '            base["args"] = sft_args')
    assert viejo in t, "No encontré la línea de SFTConfig en finetune_qlora.py. Avísame."
    t = t.replace(viejo, nuevo, 1)
    cambios.append("SFTTrainer: processing_class + dataset_text_field dentro de SFTConfig")

if "CAUSA REAL" not in t:
    viejo2 = "except (ImportError, TypeError) as exc:"
    if viejo2 in t:
        t = t.replace(viejo2, viejo2 + '\n            print("CAUSA REAL:", repr(exc))', 1)
        cambios.append("mostrar la causa real si el entrenador falla")

p.write_text(t, encoding="utf-8")
print("Cambios aplicados:" if cambios else "El parche ya estaba aplicado.", *cambios, sep="\n - ")


Cambios aplicados:
 - SFTTrainer: processing_class + dataset_text_field dentro de SFTConfig
 - mostrar la causa real si el entrenador falla


In [4]:
# 4. Dataset: combinar seed + 300, deduplicar, validar y particionar
seed = Path("training/datasets/seed.jsonl")
extra = Path("training/datasets/nitshio_dataset_300.jsonl")
combined = Path("training/datasets/train.jsonl")
assert seed.exists(), f"Falta {seed}"
assert extra.exists(), f"Falta {extra}"

raw = WORK_DIR / "combined_raw.jsonl"
raw.write_text(
    seed.read_text(encoding="utf-8").rstrip("\n") + "\n" + extra.read_text(encoding="utf-8"),
    encoding="utf-8",
)
run([sys.executable, "training/validate_dataset.py", str(raw), "--output", str(combined)])
run([sys.executable, "training/validate_dataset.py", str(combined), "--strict",
     "--split-dir", "training/datasets/splits"])
print("Dataset combinado:", combined)
print("Líneas:", len(combined.read_text(encoding="utf-8").splitlines()))


$ /usr/bin/python3 training/validate_dataset.py /content/combined_raw.jsonl --output training/datasets/train.jsonl
  NITSHIO — validación de dataset
Archivo: /content/combined_raw.jsonl
Líneas: 648 (vacías: 0)
Ejemplos válidos: 348
Errores: 0
Duplicados descartados: 300
Hash: 8e96d86b1293b107a0f22b313596fa0813b92692537b51c92a5f1d4b005a8108
Por emoción: {"angry": 58, "fun": 58, "joy": 58, "neutral": 58, "sorrow": 58, "surprised": 58}
  - línea 349 [duplicado]: duplicado exacto de la línea 49.
  - línea 350 [duplicado]: duplicado exacto de la línea 50.
  - línea 351 [duplicado]: duplicado exacto de la línea 51.
  - línea 352 [duplicado]: duplicado exacto de la línea 52.
  - línea 353 [duplicado]: duplicado exacto de la línea 53.
  - línea 354 [duplicado]: duplicado exacto de la línea 54.
  - línea 355 [duplicado]: duplicado exacto de la línea 55.
  - línea 356 [duplicado]: duplicado exacto de la línea 56.
  - línea 357 [duplicado]: duplicado exacto de la línea 57.
  - línea 358 [duplicad

In [8]:
#4.1
p = PROJECT_DIR / "training" / "finetune_qlora.py"
t = p.read_text(encoding="utf-8")
viejo = 'max_length=int(plan["max_seq_length"]),'
assert viejo in t, "No encontre max_length en SFTConfig. Avisame."
if "dataset_num_proc" not in t:
    t = t.replace(viejo, viejo + '\n                dataset_num_proc=1,', 1)
    p.write_text(t, encoding="utf-8")
    print("Parche aplicado")
else:
    print("Ya estaba aplicado")

Ya estaba aplicado


In [9]:
#4.2
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # inocuo si no aplica

p = PROJECT_DIR / "training" / "finetune_qlora.py"
t = p.read_text(encoding="utf-8")

if "_pretokenizar" not in t:
    viejo = ('    train_dataset = Dataset.from_list(train_records)\n'
             '    val_dataset = Dataset.from_list(val_records) if val_records else None\n')
    assert viejo in t, "No encontre las lineas del dataset en finetune_qlora.py. Avisame."
    nuevo = '''    def _pretokenizar(registros):
        # Tokeniza aqui para evitar el .map() de TRL, que falla al empaquetar con Unsloth.
        ids, masks = [], []
        for r in registros:
            x = list(tokenizer(r["text"], truncation=True,
                               max_length=int(plan["max_seq_length"]) - 1)["input_ids"])
            if tokenizer.eos_token_id is not None and (not x or x[-1] != tokenizer.eos_token_id):
                x.append(tokenizer.eos_token_id)
            ids.append(x)
            masks.append([1] * len(x))
        return Dataset.from_dict({"input_ids": ids, "attention_mask": masks})

    train_dataset = _pretokenizar(train_records)
    val_dataset = _pretokenizar(val_records) if val_records else None
'''
    p.write_text(t.replace(viejo, nuevo, 1), encoding="utf-8")
    print("Parche aplicado")
else:
    print("Ya estaba aplicado")

Parche aplicado


In [11]:
#4.3
p = PROJECT_DIR / "training" / "finetune_qlora.py"
t = p.read_text(encoding="utf-8")

if "dtype=None," in t:
    t = t.replace("dtype=None,", 'dtype=__import__("torch").float16,', 1)
    p.write_text(t, encoding="utf-8")
    print("Parche aplicado: modelo en float16 (T4)")
else:
    print("No encontre dtype=None; revisa si ya estaba aplicado:")
    !grep -n "dtype=" training/finetune_qlora.py

Parche aplicado: modelo en float16 (T4)


In [13]:
#4.4
import ast

p = PROJECT_DIR / "training" / "finetune_qlora.py"
t = p.read_text(encoding="utf-8")
informe = []

def aplicar(nombre, marca, viejo, nuevo):
    global t
    if marca in t:
        informe.append(f"ya estaba: {nombre}")
    elif viejo in t:
        t = t.replace(viejo, nuevo, 1)
        informe.append(f"APLICADO: {nombre}")
    else:
        informe.append(f"NO ENCONTRADO: {nombre}")

aplicar(
    "pretokenizar dataset", "_pretokenizar",
    '    train_dataset = Dataset.from_list(train_records)\n'
    '    val_dataset = Dataset.from_list(val_records) if val_records else None\n',
    '''    def _pretokenizar(registros):
        # Tokeniza aqui para evitar el .map() de TRL, que falla al empaquetar con Unsloth.
        ids, masks = [], []
        for r in registros:
            x = list(tokenizer(r["text"], truncation=True,
                               max_length=int(plan["max_seq_length"]) - 1)["input_ids"])
            if tokenizer.eos_token_id is not None and (not x or x[-1] != tokenizer.eos_token_id):
                x.append(tokenizer.eos_token_id)
            ids.append(x)
            masks.append([1] * len(x))
        return Dataset.from_dict({"input_ids": ids, "attention_mask": masks})

    train_dataset = _pretokenizar(train_records)
    val_dataset = _pretokenizar(val_records) if val_records else None
''')

aplicar("modelo en float16", 'torch").float16', "dtype=None,",
        'dtype=__import__("torch").float16,')

aplicar(
    "LoRA en float32", "_p.requires_grad",
    '        use_gradient_checkpointing="unsloth",\n'
    '        random_state=int(plan["seed"]),\n'
    '    )\n',
    '''        use_gradient_checkpointing="unsloth",
        random_state=int(plan["seed"]),
    )
    _torch = __import__("torch")
    for _p in model.parameters():
        if _p.requires_grad and _p.dtype != _torch.float32:
            _p.data = _p.data.to(_torch.float32)
''')

aplicar(
    "dataset_num_proc=1", 'kwargs["dataset_num_proc"]',
    "    kwargs = dict(comunes)\n",
    '''    kwargs = dict(comunes)
    if "dataset_num_proc" in parametros:
        kwargs["dataset_num_proc"] = 1
''')

ast.parse(t)   # si el resultado no es Python valido, no escribe nada
p.write_text(t, encoding="utf-8")
print("\n".join(informe))

APLICADO: pretokenizar dataset
APLICADO: modelo en float16
APLICADO: LoRA en float32
APLICADO: dataset_num_proc=1


In [ ]:
# 5. ENTRENAMIENTO QLoRA (salida en vivo; busca la línea 'loss' bajando)
OUTPUT_LORA = WORK_DIR / "nitshio-lora"
run([sys.executable, "training/finetune_qlora.py",
     "--dataset", str(combined),
     "--output-dir", str(OUTPUT_LORA)])
print("Adaptadores en", OUTPUT_LORA)


$ /usr/bin/python3 training/finetune_qlora.py --dataset training/datasets/train.jsonl --output-dir /content/nitshio-lora
W1110 20:40:32.083000 9987 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 20:40:32.122000 9987 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide 

In [ ]:
# 6. llama.cpp (solo convertir/cuantizar; sin CUDA) + dependencias ligeras del conversor
run([sys.executable, "-m", "pip", "install", "-q", "gguf", "sentencepiece"])

LLAMA_DIR = WORK_DIR / "llama.cpp"
if not LLAMA_DIR.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp", LLAMA_DIR])
run(["cmake", "-S", LLAMA_DIR, "-B", LLAMA_DIR / "build", "-DGGML_CUDA=OFF"])
run(["cmake", "--build", LLAMA_DIR / "build", "--config", "Release", "-j",
     "--target", "llama-quantize"])
print("llama.cpp listo en", LLAMA_DIR)


In [ ]:
# 7. Exportar a GGUF
OUTPUT_GGUF = WORK_DIR / "nitshio-gguf"
run([sys.executable, "training/export_gguf.py",
     "--adapter-dir", str(OUTPUT_LORA),
     "--llama-cpp-dir", str(LLAMA_DIR),
     "--output-dir", str(OUTPUT_GGUF)])
print("GGUF en", OUTPUT_GGUF)


In [ ]:
# 8. Localizar el GGUF y descargarlo (el .gguf directo, sin zip: ya viene comprimido)
ggufs = sorted(OUTPUT_GGUF.rglob("*.gguf"))
assert ggufs, f"No encontré ningún .gguf en {OUTPUT_GGUF}"
for g in ggufs:
    print(f"{g}  ({g.stat().st_size / 1e9:.2f} GB)")

GGUF_FINAL = ggufs[-1]
try:
    from google.colab import files
    files.download(str(GGUF_FINAL))
except Exception as e:
    print("La descarga automática falló:", e)
    print("Bájalo a mano: panel Archivos → clic derecho sobre el .gguf → Descargar.")
    print("O ejecuta la celda 9 para copiarlo a Google Drive.")


In [ ]:
# 9. (Opcional, más fiable para archivos grandes) copiar el GGUF a Google Drive
from google.colab import drive
drive.mount("/content/drive")
destino = Path("/content/drive/MyDrive/Nitshio")
destino.mkdir(parents=True, exist_ok=True)
shutil.copy2(GGUF_FINAL, destino / GGUF_FINAL.name)
print("Copiado a", destino / GGUF_FINAL.name)


## Después de descargar el GGUF (en tu PC)

```powershell
# 1) Crear nitshio-v2 SIN activarlo todavía (con tu ruta real)
.\.venv\Scripts\python.exe training\build_model.py `
  --version nitshio-v2 --gguf C:\ruta\nitshio-q4_K_M.gguf --create

# 2) Comparar v1 contra v2
.\.venv\Scripts\python.exe training\evaluate.py --models nitshio-v1 nitshio-v2 --record-metrics --no-judge
```

Lee 8 o 10 respuestas reales de v2 y compáralas con v1. Si v2 gana, actívala; si no, sigues con v1.
Pégale el reporte a Claude para interpretarlo.
